In [ ]:
# ============================================================
# NOTEBOOK 03 - Gold Layer: Star Schema
# ============================================================

# Le a tabela Silver
df = spark.table("workspace.default.silver_voos")
print("Silver carregada:", df.count(), "linhas")
print("Colunas:", df.columns)

In [ ]:
# Dimensao Tempo
spark.sql("""
CREATE OR REPLACE TABLE workspace.default.gold_dim_tempo AS
SELECT DISTINCT ANO, MS AS MES,
    CASE MS
        WHEN 1 THEN 'Janeiro'  WHEN 2 THEN 'Fevereiro'
        WHEN 3 THEN 'Marco'    WHEN 4 THEN 'Abril'
        WHEN 5 THEN 'Maio'     WHEN 6 THEN 'Junho'
        WHEN 7 THEN 'Julho'    WHEN 8 THEN 'Agosto'
        WHEN 9 THEN 'Setembro' WHEN 10 THEN 'Outubro'
        WHEN 11 THEN 'Novembro' WHEN 12 THEN 'Dezembro'
    END AS NOME_MES,
    CASE
        WHEN MS IN (12, 1, 2) THEN 'Verão'
        WHEN MS IN (3, 4, 5)  THEN 'Outono'
        WHEN MS IN (6, 7, 8)  THEN 'Inverno'
        ELSE 'Primavera'
    END AS ESTACAO
FROM workspace.default.silver_voos
""")
print("dim_tempo OK")
spark.sql("SELECT count(*) FROM workspace.default.gold_dim_tempo").show()

In [ ]:
# Coluna 4 - Tabela Fato Voos
spark.sql("""
CREATE OR REPLACE TABLE workspace.default.gold_fato_voos AS
SELECT
    ANO,
    MS AS MES,
    EMPRESA_SIGLA,
    AEROPORTO_DE_ORIGEM_SIGLA AS ORIGEM,
    AEROPORTO_DE_DESTINO_SIGLA AS DESTINO,
    SUM(PASSAGEIROS_PAGOS)     AS PASSAGEIROS_PAGOS,
    SUM(PASSAGEIROS_GRTIS)     AS PASSAGEIROS_GRTIS,
    SUM(CARGA_PAGA_KG)         AS CARGA_PAGA_KG,
    SUM(CARGA_GRTIS_KG)        AS CARGA_GRTIS_KG,
    SUM(CORREIO_KG)            AS CORREIO_KG,
    SUM(DECOLAGENS)            AS DECOLAGENS,
    SUM(ASSENTOS)              AS ASSENTOS,
    SUM(PAYLOAD)               AS PAYLOAD,
    SUM(HORAS_VOADAS)          AS HORAS_VOADAS,
    SUM(DISTNCIA_VOADA_KM)     AS DISTNCIA_VOADA_KM
FROM workspace.default.silver_voos
GROUP BY ANO, MS, EMPRESA_SIGLA,
    AEROPORTO_DE_ORIGEM_SIGLA, AEROPORTO_DE_DESTINO_SIGLA
""")
print("gold_fato_voos OK")
spark.sql("SELECT count(*) FROM workspace.default.gold_fato_voos").show()

In [ ]:
# Dimensao Empresa
spark.sql("""
CREATE OR REPLACE TABLE workspace.default.gold_dim_empresa AS
SELECT DISTINCT
    EMPRESA_SIGLA,
    EMPRESA_NOME,
    EMPRESA_NACIONALIDADE
FROM workspace.default.silver_voos
""")
print("dim_empresa OK")
spark.sql("SELECT count(*) FROM workspace.default.gold_dim_empresa").show()

In [ ]:
# Dimensao Aeroporto
spark.sql("""
CREATE OR REPLACE TABLE workspace.default.gold_dim_aeroporto AS
SELECT DISTINCT AEROPORTO_DE_ORIGEM_SIGLA AS SIGLA,
    AEROPORTO_DE_ORIGEM_NOME AS NOME,
    AEROPORTO_DE_ORIGEM_UF AS UF,
    AEROPORTO_DE_ORIGEM_REGIO AS REGIAO,
    AEROPORTO_DE_ORIGEM_PAS AS PAIS
FROM workspace.default.silver_voos
UNION
SELECT DISTINCT AEROPORTO_DE_DESTINO_SIGLA,
    AEROPORTO_DE_DESTINO_NOME,
    AEROPORTO_DE_DESTINO_UF,
    AEROPORTO_DE_DESTINO_REGIO,
    AEROPORTO_DE_DESTINO_PAS
FROM workspace.default.silver_voos
""")
print("dim_aeroporto OK")
spark.sql("SELECT count(*) FROM workspace.default.gold_dim_aeroporto").show()

In [ ]:
# Verificar todas as tabelas Gold criadas
spark.sql("SHOW TABLES IN workspace.default LIKE 'gold*'").show()